In [259]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix
from xgboost import XGBClassifier
from sklearn.model_selection import KFold


In [263]:
def one_run(X, y, model):


    X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.3,
            stratify=y
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)


    return (
        f1_score(y_test, y_pred),
        confusion_matrix(y_test, y_pred)
    )

In [280]:
def evaluate_model(X, y, model, n_runs=10):
    f1_scores = []
    conf_matrices = []

    for i in range(n_runs):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.3,
            random_state=i,
            stratify=y
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # F1
        f1_scores.append(f1_score(y_test, y_pred))

        # Confusion matrix
        conf_matrices.append(confusion_matrix(y_test, y_pred))

    #print(f"Size of training set: {len(y_train)}", f"Size of testing set: {len(y_test)}")
    # Convert to numpy array for averaging
    conf_matrices = np.array(conf_matrices)

    return (
        np.mean(f1_scores),
        np.std(f1_scores),
        np.mean(conf_matrices, axis=0),
        np.std(conf_matrices, axis=0)
    )


In [ ]:
def test_many_datasets(s_features,id_cols):
    model = XGBClassifier(
    max_depth=8,
    learning_rate=0.1,
    n_estimators=100,
    scale_pos_weight=3,
    eval_metric="logloss",
    objective="binary:logistic",
    tree_method="hist"
    )

    f1s = []
    matrices = []
    for i in range(1,10):
        dataset = pd.read_csv(f"mini_1:5_#{i}.csv")
        X = dataset.drop(columns=id_cols + ["label"] + s_features)
        y = dataset["label"]
        X = X.fillna(0)

        score, conf_matrix = one_run(X,y,model)
        f1s.append(score)
        matrices.append(conf_matrix)

    mean_f1= np.mean(f1s)
    std_f1 = np.std(f1s)
    avg_conf = np.mean(matrices,axis=0)
    std_conf = np.std(matrices,axis=0)
    print("ONLY R FEATURES + remove U-HA_R_RetweetPercent and  U-HA_R_AverageInterval")
    print("-" * 40)

    print("Mixed 1:5 for 9 different datasets (same pos but random neg):")
    print("-" * 40)

    print(f"F1: {mean_f1:.4f} ± {std_f1:.4f}")

    print("Confusion Matrix")
    print(f"TN: {avg_conf[0,0]:.1f} ± {std_conf[0,0]:.1f}")
    print(f"FP: {avg_conf[0,1]:.1f} ± {std_conf[0,1]:.1f}")
    print(f"FN: {avg_conf[1,0]:.1f} ± {std_conf[1,0]:.1f}")
    print(f"TP: {avg_conf[1,1]:.1f} ± {std_conf[1,1]:.1f}")
    print("-" * 40)


    f1s = []
    matrices = []
    for i in range(1,10):
        dataset = pd.read_csv(f"mini_1:1_#{i}.csv")
        X = dataset.drop(columns=id_cols + ["label"] + s_features)
        y = dataset["label"]
        X = X.fillna(0)

        score, conf_matrix = one_run(X,y,model)
        f1s.append(score)
        matrices.append(conf_matrix)

    mean_f1= np.mean(f1s)
    std_f1 = np.std(f1s)
    avg_conf = np.mean(matrices,axis=0)
    std_conf = np.std(matrices,axis=0)

    print("Mixed 1:1 for 9 different datasets (same pos but random neg):")
    print("-" * 40)

    print(f"F1: {mean_f1:.4f} ± {std_f1:.4f}")

    print("Confusion Matrix")
    print(f"TN: {avg_conf[0,0]:.1f} ± {std_conf[0,0]:.1f}")
    print(f"FP: {avg_conf[0,1]:.1f} ± {std_conf[0,1]:.1f}")
    print(f"FN: {avg_conf[1,0]:.1f} ± {std_conf[1,0]:.1f}")
    print(f"TP: {avg_conf[1,1]:.1f} ± {std_conf[1,1]:.1f}")
    print("-" * 40)

In [ ]:

def in_distribution_hashtags(df, n_runs=10):
    print("In-Distribution")
    results = {}
    


    model = XGBClassifier(
    max_depth=8,
    learning_rate=0.1,
    n_estimators=100,
    scale_pos_weight=3,
    reg_lambda=2,
    eval_metric="logloss",
    objective="binary:logistic",
    tree_method="hist"
    )
    
    for tag in df["hashtag"].unique():

        df_tag = df[df["hashtag"] == tag]


        X = df_tag.drop(columns=["label", "hashtag", "A_id", "S_id", "P_id"])
        y = df_tag["label"]

        mean_f1, std_f1, avg_conf = evaluate_model(X, y, model, n_runs=n_runs)

        results[tag] = (mean_f1, std_f1)

        print(f"Hashtag: {tag}")
        print(f"F1: {mean_f1:.4f} ± {std_f1:.4f}")
        print("-" * 40)



In [ ]:


def out_of_distribution_hashtags(df):

    hashtags = df["hashtag"].unique()
    print("Out-of-Distribution")
    results = {}
    

    model = XGBClassifier(
    max_depth=8,
    learning_rate=0.1,
    n_estimators=100,
    scale_pos_weight=3,
    reg_lambda=2,
    eval_metric="logloss",
    objective="binary:logistic",
    tree_method="hist"
    )
        
    for tag in hashtags:

        test_df = df[df["hashtag"] == tag].copy()
        train_df = df[df["hashtag"] != tag].copy()
        print(f"Size of training set: {len(train_df)}", f"Size of testing set: {len(test_df)}")

        X_test = test_df.drop(columns=["label", "hashtag", "A_id", "S_id", "P_id"])
        y_test = test_df["label"]

        f1_scores = []

        kf = KFold(n_splits=10, shuffle=True, random_state=42)

        for train_index, _ in kf.split(train_df):

            train_subset = train_df.iloc[train_index]

            X_train = train_subset.drop(columns=["label", "hashtag", "A_id", "S_id", "P_id"])
            y_train = train_subset["label"]

            model.fit(X_train, y_train)

            y_pred = model.predict(X_test)
            f1_scores.append(f1_score(y_test, y_pred))

        mean_f1 = np.mean(f1_scores)
        std_f1 = np.std(f1_scores)

        print(f"Hashtag: {tag}")
        print(f"F1: {mean_f1:.4f} ± {std_f1:.4f}")
        print("-" * 40)

        results[tag] = (mean_f1, std_f1)





In [278]:
df_1_1 = pd.read_csv("mini_1:1_#1.csv")
df_1_5 = pd.read_csv("mini_1:5_#1.csv")

In [147]:
out_of_distribution_hashtags(df_1_1)

Out-of-Distribution
Size of training set: 65222 Size of testing set: 2747
Hashtag: TheTraitors
F1: 0.7624 ± 0.0017
----------------------------------------
Size of training set: 58364 Size of testing set: 9605
Hashtag: Anime
F1: 0.7557 ± 0.0012
----------------------------------------
Size of training set: 56203 Size of testing set: 11766
Hashtag: art
F1: 0.7661 ± 0.0008
----------------------------------------
Size of training set: 65024 Size of testing set: 2945
Hashtag: olympics
F1: 0.7620 ± 0.0029
----------------------------------------
Size of training set: 62688 Size of testing set: 5281
Hashtag: AI
F1: 0.7628 ± 0.0007
----------------------------------------
Size of training set: 59159 Size of testing set: 8810
Hashtag: booksky
F1: 0.7633 ± 0.0012
----------------------------------------
Size of training set: 58813 Size of testing set: 9156
Hashtag: Pokemon
F1: 0.7705 ± 0.0010
----------------------------------------
Size of training set: 62621 Size of testing set: 5348
Hashtag

In [146]:
out_of_distribution_hashtags(df_1_5)

Out-of-Distribution
Size of training set: 196492 Size of testing set: 8295
Hashtag: TheTraitors
F1: 0.5095 ± 0.0040
----------------------------------------
Size of training set: 175686 Size of testing set: 29101
Hashtag: Anime
F1: 0.4861 ± 0.0012
----------------------------------------
Size of training set: 169443 Size of testing set: 35344
Hashtag: art
F1: 0.5251 ± 0.0013
----------------------------------------
Size of training set: 195934 Size of testing set: 8853
Hashtag: olympics
F1: 0.4991 ± 0.0032
----------------------------------------
Size of training set: 188826 Size of testing set: 15961
Hashtag: AI
F1: 0.5123 ± 0.0033
----------------------------------------
Size of training set: 178225 Size of testing set: 26562
Hashtag: booksky
F1: 0.5189 ± 0.0026
----------------------------------------
Size of training set: 177314 Size of testing set: 27473
Hashtag: Pokemon
F1: 0.5086 ± 0.0025
----------------------------------------
Size of training set: 188731 Size of testing set: 

In [148]:
in_distribution_hashtags(df_1_1)

In-Distribution
Size of training set: 1922 Size of testing set: 825
Hashtag: TheTraitors
F1: 0.7434 ± 0.0094
----------------------------------------
Size of training set: 6723 Size of testing set: 2882
Hashtag: Anime
F1: 0.7560 ± 0.0030
----------------------------------------
Size of training set: 8236 Size of testing set: 3530
Hashtag: art
F1: 0.7670 ± 0.0053
----------------------------------------
Size of training set: 2061 Size of testing set: 884
Hashtag: olympics
F1: 0.7418 ± 0.0114
----------------------------------------
Size of training set: 3696 Size of testing set: 1585
Hashtag: AI
F1: 0.7550 ± 0.0040
----------------------------------------
Size of training set: 6167 Size of testing set: 2643
Hashtag: booksky
F1: 0.7586 ± 0.0059
----------------------------------------
Size of training set: 6409 Size of testing set: 2747
Hashtag: Pokemon
F1: 0.7659 ± 0.0047
----------------------------------------
Size of training set: 3743 Size of testing set: 1605
Hashtag: superbowl
F1:

In [145]:
in_distribution_hashtags(df_1_5)

In-Distribution
Size of training set: 5806 Size of testing set: 2489
Hashtag: TheTraitors
F1: 0.4631 ± 0.0132
----------------------------------------
Size of training set: 20370 Size of testing set: 8731
Hashtag: Anime
F1: 0.4727 ± 0.0113
----------------------------------------
Size of training set: 24740 Size of testing set: 10604
Hashtag: art
F1: 0.5238 ± 0.0097
----------------------------------------
Size of training set: 6197 Size of testing set: 2656
Hashtag: olympics
F1: 0.4377 ± 0.0276
----------------------------------------
Size of training set: 11172 Size of testing set: 4789
Hashtag: AI
F1: 0.4900 ± 0.0066
----------------------------------------
Size of training set: 18593 Size of testing set: 7969
Hashtag: booksky
F1: 0.5058 ± 0.0100
----------------------------------------
Size of training set: 19231 Size of testing set: 8242
Hashtag: Pokemon
F1: 0.4956 ± 0.0063
----------------------------------------
Size of training set: 11239 Size of testing set: 4817
Hashtag: supe

In [ ]:



s_features = [
    "U-HA_S_MentionR",
    "U-HA_S_MentionPerR",
    "U-P_S_AccountAge",
    "U-P_S_FollowerNum",
    "U-P_S_FolloweeNum",
    "U-P_S_TweetNum",
    "U-P_S_SpreadActivity",
    "U-P_S_FollowerNumDay",
    "U-P_S_FolloweeNumDay",
    "U-P_S_TweetNumDay",
    "U-P_S_ProfileUrl",
    "U-HA_S_RetweetPercent",
    "U-HA_S_AverageInterval",
    "U-HA_S_RetweetedRate",
    "U-HA_S_QuotedRate",
    "U-HA_S_RepliedRate",
    "U-HA_S_LikedRate",
    "U-HA_S_TweetNum"
]

r_features = [
    "U-P_R_FollowS",
    "U-HA_R_MentionS",
    "U-HA_R_MentionPerS",
    "U-HA-R_repostsS",
    "U-P_R_activeBeforeP",
    "U-P_R_AccountAge",
    "U-P_R_FollowerNum",
    "U-P_R_FolloweeNum",
    "U-P_R_TweetNum",
    "U-P_R_SpreadActivity",
    "U-P_R_FollowerNumDay",
    "U-P_R_FolloweeNumDay",
    "U-P_R_TweetNumDay",
    "U-P_R_ProfileUrl",
    "U-HA_R_RetweetPercent",
    "U-HA_R_AverageInterval",
    "U-HA_R_RetweetedRate",
    "U-HA_R_QuotedRate",
    "U-HA_R_RepliedRate",
    "U-HA_R_LikedRate",
    "U-HA_R_TweetNum"
]

id_cols = ["A_id", "S_id", "P_id", "hashtag"]


model = XGBClassifier(
    max_depth=8,
    learning_rate=0.1,
    n_estimators=100,
    scale_pos_weight=3,
    eval_metric="logloss",
    objective="binary:logistic",
    tree_method="hist"
    )


print("ONLY R FEATURES")
X = df_1_5.drop(columns=id_cols + ["label"] + s_features)
y = df_1_5["label"]
X = X.fillna(0)

mean_f1, std_f1, avg_conf, std_conf = evaluate_model(X, y, model, n_runs=10)

print("Mixed 1:5:")
print("-" * 40)

print(f"F1: {mean_f1:.4f} ± {std_f1:.4f}")

print("Confusion Matrix")
print(f"TN: {avg_conf[0,0]:.1f} ± {std_conf[0,0]:.1f}")
print(f"FP: {avg_conf[0,1]:.1f} ± {std_conf[0,1]:.1f}")
print(f"FN: {avg_conf[1,0]:.1f} ± {std_conf[1,0]:.1f}")
print(f"TP: {avg_conf[1,1]:.1f} ± {std_conf[1,1]:.1f}")
print("-" * 40)



X = df_1_1.drop(columns=id_cols + ["label"] + s_features)
y = df_1_1["label"]
X = X.fillna(0)


mean_f1, std_f1, avg_conf, std_conf = evaluate_model(X, y, model, n_runs=10)

print("Mixed 1:1:")
print("-" * 40)

print(f"F1: {mean_f1:.4f} ± {std_f1:.4f}")
print("Confusion Matrix")
print(f"TN: {avg_conf[0,0]:.1f} ± {std_conf[0,0]:.1f}")
print(f"FP: {avg_conf[0,1]:.1f} ± {std_conf[0,1]:.1f}")
print(f"FN: {avg_conf[1,0]:.1f} ± {std_conf[1,0]:.1f}")
print(f"TP: {avg_conf[1,1]:.1f} ± {std_conf[1,1]:.1f}")
print("-" * 40)


Mixed 1:5:
----------------------------------------
F1: 0.5150 ± 0.0023
Confusion Matrix
TN: 43117.2 ± 124.9
FP: 8255.8 ± 124.9
FN: 3710.1 ± 59.3
TP: 6352.9 ± 59.3
----------------------------------------
Mixed 1:1:
----------------------------------------
F1: 0.7698 ± 0.0013
Confusion Matrix
TN: 5325.2 ± 38.8
FP: 4980.8 ± 38.8
FN: 661.4 ± 22.1
TP: 9431.6 ± 22.1
----------------------------------------


In [276]:

booster = model.get_booster()

# Get raw gain importance
importance_dict = booster.get_score(importance_type="gain")

# Convert to DataFrame
importance_df = pd.DataFrame(
    list(importance_dict.items()),
    columns=["feature", "gain"]
)

# Sort by gain
importance_df = importance_df.sort_values(
    by="gain",
    ascending=False
).reset_index(drop=True)

# Convert gain to percentage importance
total_gain = importance_df["gain"].sum()
importance_df["gain"] = (
    importance_df["gain"] / total_gain
)

print(importance_df)

                 feature      gain
0     U-HA_R_RepliedRate  0.157694
1        U-HA_R_TweetNum  0.156326
2      U-P_R_TweetNumDay  0.096040
3      U-HA_R_QuotedRate  0.094111
4       U-HA_R_LikedRate  0.087844
5   U-HA_R_RetweetedRate  0.078593
6         U-P_R_TweetNum  0.069512
7   U-P_R_FollowerNumDay  0.042424
8       U-P_R_AccountAge  0.038138
9   U-P_R_FolloweeNumDay  0.037427
10     U-P_R_FolloweeNum  0.034779
11     U-P_R_FollowerNum  0.031813
12      U-P_R_ProfileUrl  0.031407
13  U-P_SR_followersDiff  0.017891
14         U-P_R_FollowS  0.016862
15   U-P_R_activeBeforeP  0.009140
